# Open in Colab
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH03/ch03_TreeQuest.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# About this notebook

This notebook demonstrates how to use [**TreeQuest**](https://github.com/SakanaAI/treequest) with **AB-MCTS style search** to iteratively improve an LLM generated answer. The loop proposes an initial solution, refines it, scores each candidate via a structured judge, and lets Monte Carlo Tree Search keep the best result discovered within a small budget.

## What it shows

* **Tree-guided refinement** of an answer with a simple two move generator: initial draft, then refine
* **Structured judging** that returns a numeric quality score in `[0, 1]` using an LLM with JSON output
* **Stateful search** with `treequest.ABMCTSA`, periodic best-so-far tracking, and a final top-k selection
* **Clean separation of roles**: generator, refiner, judge, and search controller

## What you will run

1. Install `treequest[abmcts-m]` and set up an OpenAI compatible `OpenAI` client.
2. Define a minimal `State` for the MCTS nodes: the current `llm_answer` and its `score`.
3. Implement:

   * `initial_generation()` to produce the first answer and score it
   * `refine_answer()` to improve a given answer and rescore it
   * `evaluate_answer()` to judge quality with a structured response
   * `generate()` as the action used by TreeQuest to expand nodes
4. Create an `algo = tq.ABMCTSA()` and run a short search loop, printing the best candidate as you go.
5. Select and print the final best answer with `tq.top_k`.

## How it works

* **State**
  Each node stores `llm_answer` and `score`. MCTS uses the score for selection and backpropagation.

* **Initial and refine steps**
  `initial_generation()` asks the model for a first solution to the task.
  `refine_answer()` prompts the model to improve clarity and accuracy while preserving the task.

* **Scoring**
  `evaluate_answer()` asks the model to return a JSON object like `{"score": 0.92}` and parses it with a Pydantic schema. This isolates judging from generation, which often improves stability.

* **TreeQuest**
  `ABMCTSA` handles selection, expansion via `generate`, rollouts if defined, and value backpropagation. The `generate` function returns the new `State` plus a numeric value for the parent edge. We use the parent score as the edge value, which is a simple but workable choice for a refine-only loop.

* **Top-k**
  Periodically call `tq.top_k` to inspect the best candidate so far, then call it once at the end to produce the final winner.

## Extend and adapt

* Add a **self-consistency judge** that evaluates test cases or style guides.
* Use a **pairwise judge** that compares two answers and returns a preference. Convert preferences to scores for MCTS.
* Add a **rollout step** that chains multiple small refinements before scoring.
* Incorporate **domain tests** in the judge, for example run code and check outputs.
* Persist search trees and traces for analysis with a simple store.

## Requirements and notes

* Set a valid API key in `OpenAI(api_key="...")`.
* The judge model must support structured responses. The sample uses `client.chat.completions.parse` with a Pydantic schema.
* Generation temperature is moderate for diversity, judging temperature is low for stability.
* Tree search depth and steps are small in this demo. Increase them carefully since API cost scales with calls.


# Install dependencies

In [ ]:
!pip install treequest==0.2.0

# API setup

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()


OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [ ]:
from openai import OpenAI
client = OpenAI()

# Imports

In [ ]:
import json
import treequest as tq
from openai import OpenAI
from dataclasses import dataclass
from pydantic import BaseModel, Field

# Class and data

Search node payload.
- `llm_answer`: the current candidate solution
- `score`: numeric quality in [0, 1] produced by the judge

In [ ]:
@dataclass
class State:
    llm_answer: str
    score: float


# Initial generation

Creates the first candidate answer and score it.
Uses the model to draft a solution from scratch, then calls the judge to get a score. Returns a State that MCTS can treat as a root child.

In [ ]:
def initial_generation() -> State:
    prompt = "Q: Write code in Python for the Fibonacci sequence. \nA:"
    response = client.chat.completions.create(
        model="gpt-4o",  # Must be a JSON-mode-supported model
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    answer = response.choices[0].message.content.strip()
    score = evaluate_answer(answer)
    return State(llm_answer=answer, score=score)

# Refinement

Improve an existing answer and rescore it. The refine prompt asks for clarity, accuracy, and completeness. Returns a new State with the refined text and its fresh score.

In [ ]:
def refine_answer(llm_answer: str, score: float) -> State:
    prompt = f"""The current answer is:\n\n{llm_answer}\n\nPlease improve this answer to be more informative, accurate, and clear."""
    response = client.chat.completions.create(
        model="gpt-4o",  # Must be a JSON-mode-supported model
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    refined = response.choices[0].message.content.strip()
    score = evaluate_answer(refined)
    return State(llm_answer=refined, score=score)

# Judge schema
Structured judge response.
- `score`: float in [0, 1], where higher is better. Enforced via Pydantic so parsing failures are caught early.

In [ ]:
class ScoreResponse(BaseModel):
    score: float = Field(..., ge=0.0, le=1.0)

# Judging function

Ask an LLM judge to return a JSON object with a single `score` key. Uses the OpenAI client with structured parsing into `ScoreResponse`. Falls back to 0.5 on errors to keep the search moving.

In [ ]:
def evaluate_answer(answer: str) -> float:
    prompt = (
        f"Evaluate the quality of this answer on a scale from 0 to 1.\n"
        f"Return a JSON object like {{\"score\": 0.92}}.\n\n"
        f"Answer:\n{answer}"
    )

    try:
        completion = client.chat.completions.parse(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            response_format=ScoreResponse,
        )
        return completion.choices[0].message.parsed.score
    except Exception as e:
        print(f"[Evaluation error] {e}")
        return 0.5

# Generator for TreeQuest

TreeQuest expansion function.If there is no `parent_state`, create the initial candidate. Otherwise, refine the parent answer.

In [ ]:
def generate(parent_state: State | None) -> tuple[State, float]:
    if parent_state is None:
        return initial_generation(), initial_generation().score
    return refine_answer(parent_state.llm_answer, parent_state.score), parent_state.score

# Search loop

In [ ]:
# TreeQuest loop
algo = tq.ABMCTSA()
search_tree = algo.init_tree()

# Run a small number of steps. Increase gradually in real use.
for i in range(5):
    search_tree = algo.step(search_tree, {'LLM-Refine': generate})
    # Inspect best-so-far periodically
    if (i + 1) % 5 == 0:
        best, _ = tq.top_k(search_tree, algo, k=1)[0]
        print(f"[Step {i+1}] Best so far: {best.llm_answer} (score={best.score:.2f})")

# Final selection
best_state, _ = tq.top_k(search_tree, algo, k=1)[0]
print(f"\n Final Best Answer: {best_state.llm_answer} (score={best_state.score:.2f})")


[Step 5] Best so far: Certainly! Let's refine the explanation and the code to make it more informative, accurate, and clear.

### Improved Explanation

The Fibonacci sequence is a series of numbers where each number is the sum of the two preceding ones, typically starting with 0 and 1. The sequence starts as 0, 1, 1, 2, 3, 5, 8, and so on. In this code, we aim to generate the first `n` terms of the Fibonacci sequence.

The function `fibonacci(n)` is designed to return a list containing the first `n` terms of the Fibonacci sequence. Here's a step-by-step explanation:

1. **Function Definition**: The function `fibonacci(n)` takes one argument `n`, which represents the number of terms you want to generate.

2. **Initialization**: 
    - An empty list called `sequence` is initialized to store the Fibonacci numbers.
    - Two variables, `a` and `b`, are initialized to 0 and 1, representing the first two numbers in the Fibonacci sequence.

3. **While Loop**: 
    - The loop continues until t